In [ ]:
# !pip install ta-lib
# !pip install gdown
# !pip install requests
# !pip install numpy
# !pip install pandas

In [ ]:
# !pip install -r requirements_dev.txt       
# !pip install pyotp
# !pip install logzero
# !pip install websocket-client    

In [ ]:
# !pip uninstall pycrypto
# !pip install pycryptodome    

In [ ]:
import pandas as pd
# import numpy as np
import requests
from datetime import datetime
import socket
import uuid
# import http.client
import time
import requests # type: ignore
# import mimetypes
import json
import talib
# import gdown
import http
import ssl
import os

In [ ]:
# !pip install python-dotenv

In [ ]:
from SmartApi import SmartConnect #or from SmartApi.smartConnect import SmartConnect
import pyotp
from logzero import logger

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Static values
user_type = "USER"
source_id = "WEB"   
api_key = os.environ["ANG_ONE_KEY"]  
client_code = os.environ["CLIENTCODE"]
password = os.environ["PASSWORD"]
window = 965

bot_token =  os.environ["BOT_TOKEN"]
test_mode = os.environ["TEST_MODE"].lower() == 'true'


# todays_date = datetime.today().strftime("%Y-%m-%d")
todays_date = (datetime.today() - pd.DateOffset(days=0)).strftime("%Y-%m-%d")
# window_date = (datetime.today() - pd.DateOffset(days=window)).strftime("%Y-%m-%d")
window_date = datetime(2025,1,1).strftime("%Y-%m-%d")
# '2025-01-01'  # Future date to include all data

In [ ]:
local_ip = socket.gethostbyname(socket.gethostname())
smartApi = SmartConnect(api_key)

try:
    token = os.environ["TOTP_TOKEN"]
    totp = pyotp.TOTP(token).now()
except Exception as e:
    logger.error("Invalid Token: The provided token is not valid.")
    raise e


# Get Public IP
public_ip = requests.get('https://api.ipify.org').text

# Get MAC Address
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])

# Change clientcode, password, totp
payload = '''{\n\"clientcode\":\"'''+str(client_code)+'''\"
         ,\n\"password\":\"'''+str(password)+'''\"\n
		,\n\"totp\":\"'''+str(totp)+'''\"\n
    ,\n\"state\":\"Active\"\n}'''

headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key 
}


context = ssl._create_unverified_context()

conn = http.client.HTTPSConnection(
    "apiconnect.angelone.in", context=context
    )

conn.request("POST", "/rest/auth/angelbroking/user/v1/loginByPassword", payload, headers)

res = conn.getresponse()
data = res.read()
data = data.decode("utf-8")

In [ ]:
temp = json.loads(data)
jwtToken = temp["data"]["jwtToken"]
# print(jwtToken)


local_ip = socket.gethostbyname(socket.gethostname())
public_ip = requests.get('https://api.ipify.org').text
mac_address = ':'.join(['{:02x}'.format((uuid.getnode() >> ele) & 0xff)
                        for ele in range(0,8*6,8)][::-1])
authToken = f'Bearer {jwtToken}'


headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    "X-UserType": user_type,
    "X-SourceID": source_id,
    "X-ClientLocalIP": local_ip,
    "X-ClientPublicIP": public_ip,
    "X-MACAddress": mac_address,
    'X-PrivateKey': api_key,
    'Authorization': authToken ,
}


In [ ]:
# shareable_link = 'https://drive.google.com/file/d/1PdYMxjWQ4tBJp4Mmp1LjLOR2H2vkZ6on/view?usp=sharing'

# Extract the file ID
# file_id = shareable_link.split('/d/')[1].split('/view')[0]

# Construct the download URL
# download_url = f'https://drive.google.com/uc?id={file_id}'


# # Download the file using gdown
# output_file = 'Nifty500-token.csv'

# stock_symbols_df = pd.read_csv(output_file)
# stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
# stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'PO1_Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')

# main_df.to_csv('Main_df.csv', index=False)

In [ ]:

# Download the file using gdown
output_file = 'Nifty500-token.csv'
stock_symbols_df = pd.read_csv(output_file)
stock_symbols_df["token"] = stock_symbols_df["token"].fillna(891)
stock_symbols_df["token"] = stock_symbols_df["token"].astype(int)
# stocks_to_consider = 'Stocks.csv'

# stock_lists = pd.read_csv(stocks_to_consider)
# full_main_df = pd.merge(stock_lists[['rsi','symbol','win_ratio','priority']], stock_symbols_df[['Symbol','token']], left_on ='symbol' , right_on='Symbol', how='inner')


# # full_main_df.to_csv('Full_Main_df.csv', index=False)

main_df = stock_symbols_df.copy()

# full_main_df = pd.read_csv('Full_Main_df.csv')

In [ ]:
# Step 2: Function to fetch daily candle data from API
def fetch_candle_data(symbol,interval='ONE_DAY'):

    all_data = []
    start_date = window_date
    print(symbol, start_date)
    end_date = todays_date

    while start_date < end_date:
        time.sleep(0.4)  # To avoid hitting API rate limits
        chunk_end = (pd.to_datetime(start_date) + pd.DateOffset(days=100)).strftime("%Y-%m-%d")
        print(f"Fetching: {start_date}  →  {chunk_end}")
        
        payload = '''{\r\n     \"exchange\": \"NSE\",\r\n
          \"symboltoken\": \"'''+str(symbol)+'''\",\r\n     \"interval\": \"'''+str(interval)+'''\",\r\n
          \"fromdate\": \"'''+str(start_date)+''' 09:15\",\r\n     \"todate\": \"'''+str(chunk_end)+''' 16:30\"\r\n}
    '''

        conn = http.client.HTTPSConnection("apiconnect.angelone.in", context=context)
        conn.request("POST", "/rest/secure/angelbroking/historical/v1/getCandleData", payload, headers)
        res = conn.getresponse()
        data = res.read()
        data = data.decode("utf-8")
        json_data = json.loads(data)
        json_data = json_data['data']
       
        
        if not json_data:
            print(f"No data returned for {start_date} to {chunk_end}.")
            raise json_data['message']
            break
        all_data.extend(json_data)
        

        start_date = (pd.to_datetime(chunk_end)  + pd.DateOffset(days=1)).strftime("%Y-%m-%d")

    # if all_data:
        # print(all_data)
    return all_data

    

    

In [ ]:
# daily_json_data = fetch_candle_data('395','FIVE_MINUTE')
# df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
#         # break

#         # Convert 'Date' column to datetime
# df['Date'] = pd.to_datetime(df['Date'])

#         # Sort data by date in ascending order
# df = df.sort_values('Date').reset_index(drop=True)

# df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)

# df.to_csv('test_data.csv')

In [38]:
# Prepare output DataFrame
error_data = []

final_df = pd.DataFrame()
print(main_df.columns.tolist())

# Step 5: Process each stock
#iterate only first 5 rows from 2nd row ingnore 1st

# for _, row in main_df.head(50).iterrows():
for _, row in main_df.iloc[505:].iterrows():

    # time.sleep(0.4)

    name = row['Symbol']
    token = row['token']
    print(f"Processing {name} with token {token}")
    try:   
        # Fetch daily data
        daily_json_data = fetch_candle_data(token,'FIVE_MINUTE')
        
        if daily_json_data == None:
            error_data.append({
               'Name': name,
               'Token': token,
               'reasone': 'Error while fetching data'
             })    
            continue
        
        # logger.info(f"Processing {name} with token {token} and RSI {rsi}")
        
        df = pd.DataFrame(daily_json_data, columns=["Date", "Open", "High", "Low", "Close", "Volume"])
        
        # break

        # Convert 'Date' column to datetime
        df['Date'] = pd.to_datetime(df['Date'])

        # Sort data by date in ascending order
        df = df.sort_values('Date').reset_index(drop=True)

        df['RSI_14'] = talib.RSI(df['Close'], timeperiod=14)
        # keep 2 numbers after decimal  
        df['RSI_14'] = df['RSI_14'].round(2)
        df.set_index('Date', inplace=True)
        df['Token'] = token
        df['Symbol'] = name
        
        # final_df = pd.concat([final_df, df])
        
        # last_2_rsi_daily = df['RSI_14'].dropna().tail(2).values      
        last_dats = daily_json_data[-1][0].split('T')[0]
        
        df.to_csv(f'5min data/Final_5min_RSI_{name}_from_{window_date}_to_{last_dats}.csv')
        

    except Exception as e:
        error_data.append({
            'Name': name,
            'Token': token,
            'reasone': str(e)
        })
        print(f"Error processing {name}: {e}")

['Symbol', 'NAME OF COMPANY', 'token']


In [4]:
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pandas as pd

In [ ]:


# # Folder containing CSV files
# FOLDER_PATH = "test_fol"
# rsi_column = "RSI_14"


# # -----------------------------
# # CHECK PROFIT/LOSS FUNCTION
# # -----------------------------
# def check_profit_loss(df, start_index):
#     entry_price = df.loc[start_index, "Close"]
#     target_profit = entry_price * 1.01     # +1%
#     target_loss   = entry_price * 0.99     # -1%

#     future = df.loc[start_index+1:]
#     curr_date = pd.to_datetime(df.loc[start_index, "Date"]).strftime("%Y-%m-%d")

#     for idx, row in future.iterrows():
#         high = row["High"]
#         low = row["Low"]
#         date_ind = pd.to_datetime(row["Date"]).strftime("%Y-%m-%d")
#         if date_ind != curr_date:
#             break

#         if low <= target_loss:
#             return "LOSS"

#         if high >= target_profit:
#             return "WIN"

#     return "NO_RESULT" 


# # -----------------------------
# # PROCESS SINGLE CSV FILE
# # -----------------------------
# def process_file(file_path):
#     file_name = os.path.basename(file_path)

#     try:
#         df = pd.read_csv(file_path)

#         df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
#         df["High"]  = pd.to_numeric(df["High"], errors="coerce")
#         df["Low"]   = pd.to_numeric(df["Low"], errors="coerce")
#         df[rsi_column] = pd.to_numeric(df[rsi_column], errors="coerce")

#         df.dropna(subset=["Close", "High", "Low", rsi_column], inplace=True)

#         results_local = []
#         rsi_values = sorted(df[rsi_column].unique())
#         print(rsi_values)
        
        
        
        
        
        
        

#         # Loop for each RSI value
#         for rsi_value in rsi_values:
#             matching_rows = df[df[rsi_column] == rsi_value]

#             for index in matching_rows.index:
#                 result = check_profit_loss(df, index)
#                 results_local.append({
#                     "file": file_name,
#                     "rsi": rsi_value,
#                     "index": index,
#                     "result": result
#                 })

#         return results_local

#     except Exception as e:
#         print(f"Error processing {file_name}: {e}")
#         return []



In [5]:


# Folder containing CSV files
FOLDER_PATH = "test_fol"
rsi_column = "RSI_14"


# -----------------------------
# CHECK PROFIT/LOSS FUNCTION
# -----------------------------
def check_profit_loss(df, start_index):
    entry_price = df.loc[start_index, "Close"]
    target_profit = entry_price * 1.01     # +1%
    target_loss   = entry_price * 0.99     # -1%

    future = df.loc[start_index+1:]
    curr_date = pd.to_datetime(df.loc[start_index, "Date"]).strftime("%Y-%m-%d")

    for idx, row in future.iterrows():
        high = row["High"]
        low = row["Low"]
        date_ind = pd.to_datetime(row["Date"]).strftime("%Y-%m-%d")
        if date_ind != curr_date:
            break

        if low <= target_loss:
            return "LOSS"

        if high >= target_profit:
            return "WIN"

    return "NO_RESULT" 


# -----------------------------
# PROCESS SINGLE CSV FILE
# -----------------------------
def process_file(file_path):
    file_name = os.path.basename(file_path)

    try:
        df = pd.read_csv(file_path)

        df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
        df["High"]  = pd.to_numeric(df["High"], errors="coerce")
        df["Low"]   = pd.to_numeric(df["Low"], errors="coerce")
        df[rsi_column] = pd.to_numeric(df[rsi_column], errors="coerce")

        df.dropna(subset=["Close", "High", "Low", rsi_column], inplace=True)

        results_local = []
        rsi_values = sorted(df[rsi_column].unique())
        print(rsi_values)
        
        def process_single_rsi(rsi_value):
            local_results = []
            matching_rows = df[df[rsi_column] == rsi_value]

            for index in matching_rows.index:
                result = check_profit_loss(df, index)
                local_results.append({
                    "file": file_name,
                    "rsi": rsi_value,
                    "index": index,
                    "result": result
                })
            return local_results
                    
        with ProcessPoolExecutor() as tpool:
            inner_futures = tpool.map(process_single_rsi, rsi_values)

        # Combine all returned results
        for part in inner_futures:
            results_local.extend(part)
        
        
        
        



        return results_local

    except Exception as e:
        print(f"Error processing {file_name}: {e}")
        return []




In [6]:
process_file('test_fol/Final_5min_RSI_CIPLA_from_2025-01-01_to_2025-11-19.csv')

[np.float64(6.44), np.float64(7.06), np.float64(7.44), np.float64(7.57), np.float64(7.85), np.float64(8.04), np.float64(8.9), np.float64(9.59), np.float64(10.74), np.float64(12.11), np.float64(12.76), np.float64(13.12), np.float64(13.58), np.float64(13.79), np.float64(13.88), np.float64(14.19), np.float64(14.32), np.float64(14.8), np.float64(14.83), np.float64(14.84), np.float64(14.93), np.float64(14.96), np.float64(15.03), np.float64(15.2), np.float64(15.33), np.float64(15.51), np.float64(15.62), np.float64(15.67), np.float64(15.69), np.float64(15.7), np.float64(15.76), np.float64(15.89), np.float64(15.92), np.float64(16.08), np.float64(16.18), np.float64(16.29), np.float64(16.31), np.float64(16.38), np.float64(16.61), np.float64(16.63), np.float64(16.64), np.float64(16.72), np.float64(16.83), np.float64(17.01), np.float64(17.03), np.float64(17.04), np.float64(17.09), np.float64(17.1), np.float64(17.27), np.float64(17.29), np.float64(17.42), np.float64(17.47), np.float64(17.49), np.fl

[]

In [22]:
# -----------------------------
# MAIN PARALLEL EXECUTION
# -----------------------------

csv_files = glob.glob(os.path.join(FOLDER_PATH, "*.csv"))
print(f"\nTotal files found: {len(csv_files)}\n")

all_results = []

# Run in parallel
with ProcessPoolExecutor() as executor:
    futures = {executor.submit(process_file, f): f for f in csv_files}

    for future in as_completed(futures):
        file = futures[future]
        try:
            result = future.result()
            all_results.extend(result)
            print(f"Finished: {os.path.basename(file)}")
        except Exception as e:
            print(f"Error in {file}: {e}")

# Convert to DataFrame
results_df = pd.DataFrame(all_results)

print("\n\n--- SUMMARY ---")
print(results_df["result"].value_counts())

results_df.to_csv("parallel_rsi_profit_loss_output.csv", index=False)
print("\nSaved → parallel_rsi_profit_loss_output.csv")







Total files found: 1

Error in test_fol\Final_5min_RSI_CIPLA_from_2025-01-01_to_2025-11-19.csv: A process in the process pool was terminated abruptly while the future was running or pending.


--- SUMMARY ---


KeyError: 'result'

In [23]:
print(all_results)

[]
